# AI Human Clone: apni aawaz GPU par (muft)

Ye notebook aap ki **apni aawaz** banane wali service Google Colab ke muft GPU par chalata hai.
Laptop ke CPU par 1-2 minute lagte hain, GPU par chand second.

**Har baar:**
1. Upar menu: **Runtime → Change runtime type → T4 GPU → Save** (sirf pehli baar).
2. **Runtime → Run all**.
3. Google Drive ki ijazat maange to de dein.
4. Token maange to laptop wali script (`scripts\voice-gpu.ps1`) ne jo copy kiya hai wo **Ctrl+V** se paste kar ke Enter.
5. Aakhir mein jo `https://....trycloudflare.com` link aaye, use laptop wali script mein paste karein.

Ye tab khula rehne dein. Band karne par app khud laptop wali (dheemi) aawaz par chali jati hai.

Engine: OmniVoice (asal Urdu, sirf apne istemal ke liye). Pehli dafa har aawaz ki likhai
Whisper se khud banti hai aur Drive mein save ho jati hai (`<id>.txt`, `<id>.omnivoice.pt`).

**Pehli baar:** Google Drive mein `ai-clone-voices` naam ka folder banayein aur laptop ke
`Desktop\AI-Clone-GPU\voices` ki saari `.wav` files us mein upload kar dein.

In [ ]:
# 1) GPU aur Google Drive
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("GPU nahi mila! Runtime > Change runtime type > T4 GPU chunein, phir Run all dobara.")
print("GPU:", gpu)

from google.colab import drive
drive.mount("/content/drive")
VOICES = "/content/drive/MyDrive/ai-clone-voices"
os.makedirs(VOICES, exist_ok=True)
voices = sorted(f for f in os.listdir(VOICES) if f.endswith(".wav"))
print(f"Drive voices folder: {VOICES} ({len(voices)} aawaz files)")
if not voices:
    print("Abhi koi .wav nahi: laptop ke Desktop\\AI-Clone-GPU\\voices ki files is folder mein upload karein,"
          " ya app mein nayi aawaz record karein (wo yahin save hogi).")

In [ ]:
# 2) Engine aur zaroori packages (2-3 minute). Colab ka apna GPU wala PyTorch istemal hota hai.
# "omnivoice": asal Urdu bolta hai (sirf apne istemal ke liye: CC-BY-NC license).
# "chatterbox": MIT license (bechne ke liye theek), Urdu ko Hindi ke zariye bolta hai.
ENGINE = "omnivoice"

!pip -q install fastapi==0.142.2 uvicorn==0.54.0 python-multipart==0.0.32 soundfile soxr
if ENGINE == "omnivoice":
    !pip -q install omnivoice==0.2.1
else:
    !pip -q install --no-deps chatterbox-tts==0.1.7
    !pip -q install librosa==0.11.0 safetensors s3tokenizer==0.3.0 transformers==5.2.0 diffusers==0.29.0 resemble-perth==1.0.1 conformer==0.3.2 spacy-pkuseg==1.0.1 pykakasi==2.3.0 omegaconf==2.3.1 einops==0.8.2
print("Packages tayyar:", ENGINE)

In [ ]:
# 3) Voice service ka code (laptop wala hi code, build_notebook.py ne yahan daala hai)
import os
os.makedirs("/content/vc", exist_ok=True)
FILES = {
    'app.py': '"""Voice clone service: internal HTTP API around Chatterbox Multilingual.\n\nEndpoints (JSON errors as {"detail": ...}):\n  GET    /health                 -> {"status", "model_loaded", "device"}\n  POST   /voices                 multipart "files" -> 201 {"voice_id", "reference_seconds"}\n  DELETE /voices/{voice_id}      -> 204\n  POST   /synthesize             {"voice_id", "text", "language_id"} -> audio/wav\n\nLogs carry sizes and timings only; request text is never logged.\n"""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport hmac\nimport logging\nimport os\nimport re\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\nfrom typing import Callable\n\nfrom fastapi import FastAPI, File, Form, HTTPException, Request, Response, UploadFile\nfrom fastapi.concurrency import run_in_threadpool\nfrom fastapi.responses import JSONResponse\nfrom pydantic import BaseModel, Field, field_validator\n\nfrom audio import REFERENCE_SAMPLE_RATE, AudioError, build_reference, decode_audio, encode_wav\nfrom model import (\n    SUPPORTED_LANGUAGES,\n    Engine,\n    ModelManager,\n    ModelNotReady,\n    SynthesisCancelled,\n    engine_class,\n)\n\nlogging.basicConfig(\n    level=os.getenv("LOG_LEVEL", "INFO").upper(),\n    format="%(asctime)s %(levelname)s %(name)s: %(message)s",\n)\nlogger = logging.getLogger("voice_clone")\n\nVOICE_ID_RE = re.compile(r"^[0-9a-f]{32}$")\nLANGUAGE_RE = re.compile(r"^[a-z]{2,3}$")\n# Files kept next to <voice_id>.wav: its transcript and OmniVoice\'s saved clone prompt.\nVOICE_SIDECARS = (".txt", ".omnivoice.pt")\nMAX_TEXT_CHARS = 1000\nMAX_FILES = 20\n\n\ndef _env_float(name: str, default: float) -> float:\n    try:\n        return float(os.getenv(name, "") or default)\n    except ValueError:\n        return default\n\n\nclass TranscriptRequest(BaseModel):\n    text: str = Field(min_length=1, max_length=2000)\n\n\nclass SynthesizeRequest(BaseModel):\n    voice_id: str\n    text: str = Field(min_length=1, max_length=MAX_TEXT_CHARS)\n    language_id: str\n\n    @field_validator("text")\n    @classmethod\n    def _not_blank(cls, value: str) -> str:\n        if not value.strip():\n            raise ValueError("text must not be blank")\n        return value\n\n\nclass VoiceStore:\n    def __init__(self, directory: Path) -> None:\n        self.directory = directory\n\n    def path(self, voice_id: str) -> Path:\n        if not VOICE_ID_RE.fullmatch(voice_id):\n            raise HTTPException(status_code=400, detail="invalid voice_id")\n        return self.directory / f"{voice_id}.wav"\n\n    def save(self, voice_id: str, wav_bytes: bytes) -> None:\n        target = self.path(voice_id)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        tmp = target.with_suffix(".tmp")\n        tmp.write_bytes(wav_bytes)\n        os.replace(tmp, target)\n\n\ndef create_app(\n    engine_factory: Callable[[], Engine] | None = None,\n    data_dir: str | os.PathLike | None = None,\n) -> FastAPI:\n    token = os.getenv("VOICE_CLONE_SERVICE_TOKEN", "")\n    max_upload_bytes = int(_env_float("MAX_UPLOAD_MB", 50) * 1024 * 1024)\n    synth_timeout = _env_float("SYNTH_TIMEOUT_SECONDS", 240)\n    model_wait = _env_float("MODEL_WAIT_SECONDS", 30)\n    max_queue = int(_env_float("MAX_QUEUED_SYNTHESES", 4))\n\n    store = VoiceStore(Path(data_dir or os.getenv("VOICE_DATA_DIR", "/data/voices")))\n    engine_factory = engine_factory or engine_class(os.getenv("VOICE_CLONE_ENGINE", "chatterbox"))\n    engine_name = getattr(engine_factory, "name", "chatterbox")\n    # None = any language code the engine knows (OmniVoice); otherwise a fixed set.\n    languages = getattr(engine_factory, "languages", SUPPORTED_LANGUAGES)\n    manager = ModelManager(engine_factory)\n    executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="synth")\n    queue_lock = threading.Lock()\n    queued = {"count": 0}\n\n    @asynccontextmanager\n    async def lifespan(_: FastAPI):\n        manager.start_loading()\n        yield\n        executor.shutdown(wait=False, cancel_futures=True)\n\n    app = FastAPI(title="Voice Clone Service", lifespan=lifespan, docs_url=None, redoc_url=None)\n    app.state.manager = manager\n    app.state.store = store\n\n    @app.middleware("http")\n    async def guard(request: Request, call_next):\n        if token and not (request.method == "GET" and request.url.path == "/health"):\n            sent = request.headers.get("x-service-token", "")\n            if not hmac.compare_digest(sent.encode(), token.encode()):\n                return JSONResponse(status_code=401, content={"detail": "invalid service token"})\n        length = request.headers.get("content-length")\n        if length and length.isdigit() and int(length) > max_upload_bytes:\n            return JSONResponse(status_code=413, content={"detail": "upload too large"})\n        return await call_next(request)\n\n    @app.get("/health")\n    async def health():\n        return {"status": "ok", "model_loaded": manager.loaded, "device": manager.device, "engine": engine_name}\n\n    @app.post("/voices", status_code=201)\n    async def create_voice(files: list[UploadFile] = File(...), voice_id: str | None = Form(None)):\n        # A caller may choose the id, so the same voice can live on two services (GPU + laptop).\n        if voice_id is not None and not VOICE_ID_RE.fullmatch(voice_id):\n            raise HTTPException(status_code=400, detail="invalid voice_id")\n        chosen_id = voice_id\n        if len(files) > MAX_FILES:\n            raise HTTPException(status_code=400, detail=f"at most {MAX_FILES} files")\n        started = time.monotonic()\n        blobs: list[bytes] = []\n        total = 0\n        for upload in files:\n            data = await upload.read()\n            total += len(data)\n            if total > max_upload_bytes:\n                raise HTTPException(status_code=413, detail="upload too large")\n            blobs.append(data)\n\n        def process() -> tuple[str, float]:\n            clips = [decode_audio(blob) for blob in blobs]\n            reference = build_reference(clips)\n            voice_id = chosen_id or uuid.uuid4().hex\n            store.save(voice_id, encode_wav(reference, REFERENCE_SAMPLE_RATE))\n            return voice_id, reference.size / REFERENCE_SAMPLE_RATE\n\n        try:\n            voice_id, seconds = await run_in_threadpool(process)\n        except AudioError as exc:\n            raise HTTPException(status_code=400, detail=str(exc)) from exc\n        logger.info(\n            "voice created: %d file(s), %d bytes -> %.1fs reference in %.2fs",\n            len(blobs), total, seconds, time.monotonic() - started,\n        )\n        return {"voice_id": voice_id, "reference_seconds": round(seconds, 2)}\n\n    @app.delete("/voices/{voice_id}", status_code=204)\n    async def delete_voice(voice_id: str):\n        path = store.path(voice_id)\n        if not path.is_file():\n            raise HTTPException(status_code=404, detail="voice not found")\n        path.unlink(missing_ok=True)\n        for suffix in VOICE_SIDECARS:\n            path.with_suffix(suffix).unlink(missing_ok=True)\n        manager.forget_voice(voice_id)\n        logger.info("voice deleted")\n        return Response(status_code=204)\n\n    @app.get("/voices/{voice_id}/reference")\n    async def voice_reference(voice_id: str):\n        """The stored reference WAV, so the caller can transcribe it (OmniVoice needs the text)."""\n        path = store.path(voice_id)\n        if not path.is_file():\n            raise HTTPException(status_code=404, detail="voice not found")\n        return Response(content=path.read_bytes(), media_type="audio/wav")\n\n    @app.put("/voices/{voice_id}/transcript", status_code=204)\n    async def set_transcript(voice_id: str, body: TranscriptRequest):\n        """What is said in the reference. Replaces OmniVoice\'s own Whisper guess; the saved\n        clone prompt is dropped so the next speech rebuilds it with this text."""\n        path = store.path(voice_id)\n        if not path.is_file():\n            raise HTTPException(status_code=404, detail="voice not found")\n        path.with_suffix(".txt").write_text(body.text.strip(), encoding="utf-8")\n        path.with_suffix(".omnivoice.pt").unlink(missing_ok=True)\n        manager.forget_voice(voice_id)\n        logger.info("voice transcript set (%d chars)", len(body.text))\n        return Response(status_code=204)\n\n    @app.post("/synthesize")\n    async def synthesize(body: SynthesizeRequest):\n        language_id = body.language_id.strip().lower()\n        if not LANGUAGE_RE.fullmatch(language_id) or (languages is not None and language_id not in languages):\n            raise HTTPException(status_code=400, detail="unsupported language_id")\n        voice_path = store.path(body.voice_id)\n        if not voice_path.is_file():\n            raise HTTPException(status_code=404, detail="voice not found")\n        if not await run_in_threadpool(manager.wait_ready, model_wait):\n            raise HTTPException(status_code=503, detail="model loading")\n\n        with queue_lock:\n            if queued["count"] >= max_queue:\n                raise HTTPException(status_code=503, detail="busy, try again later")\n            queued["count"] += 1\n\n        cancel = threading.Event()\n        loop = asyncio.get_running_loop()\n        future = loop.run_in_executor(\n            executor, manager.synthesize, body.text, language_id, body.voice_id, voice_path, cancel\n        )\n\n        def _release(_):\n            with queue_lock:\n                queued["count"] -= 1\n\n        future.add_done_callback(_release)\n        try:\n            audio = await asyncio.wait_for(future, timeout=synth_timeout)\n        except asyncio.TimeoutError as exc:\n            cancel.set()\n            logger.warning("synthesis timed out after %.0fs (%d chars)", synth_timeout, len(body.text))\n            raise HTTPException(status_code=504, detail="synthesis timed out") from exc\n        except SynthesisCancelled as exc:\n            raise HTTPException(status_code=504, detail="synthesis timed out") from exc\n        except ModelNotReady as exc:\n            raise HTTPException(status_code=503, detail="model loading") from exc\n        except Exception as exc:\n            logger.error("synthesis failed: %s", type(exc).__name__)\n            raise HTTPException(status_code=500, detail="synthesis failed") from exc\n\n        if audio.size == 0:\n            raise HTTPException(status_code=500, detail="synthesis produced no audio")\n        wav = await run_in_threadpool(encode_wav, audio, manager.sample_rate)\n        return Response(content=wav, media_type="audio/wav")\n\n    return app\n\n\napp = create_app()\n',
    'model.py': '"""Model wrapper: lazy background loading and serialized generation.\n\nThe real Chatterbox engine imports torch only inside ``load()``, so this\nmodule (and the API) can be imported and tested without torch or weights.\n"""\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport threading\nimport time\nfrom collections import OrderedDict\nfrom pathlib import Path\nfrom typing import Callable, Protocol\n\nimport numpy as np\n\nfrom audio import split_sentences\n\nlogger = logging.getLogger("voice_clone.model")\n\nSUPPORTED_LANGUAGES = frozenset(\n    "ar da de el en es fi fr he hi it ja ko ms nl no pl pt ru sv sw tr zh".split()\n)\n\nSENTENCE_GAP_SECONDS = 0.15\n\n\nclass Engine(Protocol):\n    sample_rate: int\n    device: str\n\n    def load(self) -> None: ...\n\n    def generate(self, text: str, language_id: str, voice_id: str, voice_path: Path) -> np.ndarray:\n        """Return mono float32 samples at ``sample_rate``."""\n        ...\n\n    def forget_voice(self, voice_id: str) -> None: ...\n\n\nclass SynthesisCancelled(Exception):\n    """The request timed out; remaining sentences are skipped."""\n\n\nclass ModelNotReady(Exception):\n    pass\n\n\ndef pick_device(setting: str, cuda_available: bool) -> str:\n    """VOICE_CLONE_DEVICE: "auto" (default) uses an NVIDIA GPU when there is one\n    (e.g. on Google Colab, many times faster), otherwise the CPU."""\n    setting = (setting or "auto").strip().lower()\n    if setting == "auto":\n        return "cuda" if cuda_available else "cpu"\n    if setting == "cuda" and not cuda_available:\n        raise RuntimeError("VOICE_CLONE_DEVICE=cuda but no CUDA GPU is available")\n    return setting\n\n\nclass ChatterboxEngine:\n    """Chatterbox Multilingual (MIT). Every output carries a Perth watermark."""\n\n    name = "chatterbox"\n    languages: frozenset[str] | None = SUPPORTED_LANGUAGES\n    device = "cpu"\n\n    def __init__(self, conds_cache_size: int = 4) -> None:\n        self._model = None\n        self.sample_rate = 24_000\n        self._conds: OrderedDict[str, object] = OrderedDict()\n        self._conds_cache_size = conds_cache_size\n\n    def load(self) -> None:\n        import torch  # noqa: PLC0415 - heavy import on purpose deferred\n        from chatterbox.mtl_tts import ChatterboxMultilingualTTS  # noqa: PLC0415\n\n        threads = int(os.getenv("TORCH_NUM_THREADS", "0") or 0)\n        if threads > 0:\n            torch.set_num_threads(threads)\n        self.device = pick_device(os.getenv("VOICE_CLONE_DEVICE", "auto"), torch.cuda.is_available())\n        self._model = ChatterboxMultilingualTTS.from_pretrained(device=self.device)\n        self.sample_rate = int(self._model.sr)\n\n    def _conditionals(self, voice_id: str, voice_path: Path):\n        conds = self._conds.get(voice_id)\n        if conds is not None:\n            self._conds.move_to_end(voice_id)\n            return conds\n        self._model.prepare_conditionals(str(voice_path))\n        conds = self._model.conds\n        self._conds[voice_id] = conds\n        while len(self._conds) > self._conds_cache_size:\n            self._conds.popitem(last=False)\n        return conds\n\n    def generate(self, text: str, language_id: str, voice_id: str, voice_path: Path) -> np.ndarray:\n        # Speaker conditionals are computed once per voice and reused for every\n        # sentence instead of re-reading the reference on each generate() call.\n        self._model.conds = self._conditionals(voice_id, voice_path)\n        wav = self._model.generate(text, language_id=language_id)\n        return wav.squeeze(0).detach().cpu().numpy().astype(np.float32)\n\n    def forget_voice(self, voice_id: str) -> None:\n        self._conds.pop(voice_id, None)\n\n\nclass OmniVoiceEngine:\n    """OmniVoice (k2-fsa): 600+ languages including Urdu ("ur"). Meant for an NVIDIA GPU\n    (e.g. Google Colab); far too slow on a laptop CPU. The weights are CC-BY-NC:\n    personal / non-commercial use only.\n\n    A voice needs the transcript of its reference: ``<voice_id>.txt`` next to the WAV if\n    present, otherwise OmniVoice transcribes it with Whisper (downloaded on first use).\n    The resulting clone prompt is saved as ``<voice_id>.omnivoice.pt`` and reused."""\n\n    name = "omnivoice"\n    languages: frozenset[str] | None = None  # any language code OmniVoice knows\n    device = "cpu"\n\n    def __init__(self, prompt_cache_size: int = 4) -> None:\n        self._model = None\n        self.sample_rate = 24_000\n        self._prompts: OrderedDict[str, object] = OrderedDict()\n        self._prompt_cache_size = prompt_cache_size\n        self._config = None\n\n    def load(self) -> None:\n        import torch  # noqa: PLC0415 - heavy imports on purpose deferred\n        from omnivoice import OmniVoice, OmniVoiceGenerationConfig  # noqa: PLC0415\n\n        self.device = pick_device(os.getenv("VOICE_CLONE_DEVICE", "auto"), torch.cuda.is_available())\n        on_gpu = self.device == "cuda"\n        self._model = OmniVoice.from_pretrained(\n            os.getenv("OMNIVOICE_MODEL", "k2-fsa/OmniVoice"),\n            device_map="cuda:0" if on_gpu else "cpu",\n            dtype=torch.float16 if on_gpu else torch.float32,\n        )\n        self.sample_rate = int(self._model.sampling_rate)\n        self._config = OmniVoiceGenerationConfig(num_step=int(os.getenv("OMNIVOICE_STEPS", "32")))\n\n    def _prompt(self, voice_id: str, voice_path: Path):\n        from omnivoice import VoiceClonePrompt  # noqa: PLC0415\n\n        prompt = self._prompts.get(voice_id)\n        if prompt is not None:\n            self._prompts.move_to_end(voice_id)\n            return prompt\n        saved = voice_path.with_suffix(".omnivoice.pt")\n        if saved.is_file():\n            prompt = VoiceClonePrompt.load(str(saved))\n        else:\n            transcript = voice_path.with_suffix(".txt")\n            ref_text = transcript.read_text(encoding="utf-8").strip() if transcript.is_file() else None\n            prompt = self._model.create_voice_clone_prompt(str(voice_path), ref_text=ref_text or None)\n            try:\n                prompt.save(str(saved))\n            except OSError:\n                logger.warning("could not save the voice prompt; it is rebuilt next time")\n        self._prompts[voice_id] = prompt\n        while len(self._prompts) > self._prompt_cache_size:\n            self._prompts.popitem(last=False)\n        return prompt\n\n    def generate(self, text: str, language_id: str, voice_id: str, voice_path: Path) -> np.ndarray:\n        audio = self._model.generate(\n            text=text,\n            language=language_id,\n            voice_clone_prompt=self._prompt(voice_id, voice_path),\n            generation_config=self._config,\n        )[0]\n        return np.asarray(audio, dtype=np.float32).reshape(-1)\n\n    def forget_voice(self, voice_id: str) -> None:\n        self._prompts.pop(voice_id, None)\n\n\nENGINES = {engine.name: engine for engine in (ChatterboxEngine, OmniVoiceEngine)}\n\n\ndef engine_class(name: str):\n    """VOICE_CLONE_ENGINE: "chatterbox" (default, MIT) or "omnivoice" (GPU, non-commercial)."""\n    key = (name or "chatterbox").strip().lower()\n    if key not in ENGINES:\n        raise ValueError(f"unknown VOICE_CLONE_ENGINE {name!r} (use: {\', \'.join(ENGINES)})")\n    return ENGINES[key]\n\n\nclass ModelManager:\n    """Loads the engine in a background thread and runs one generation at a time."""\n\n    retry_initial_seconds = 15.0\n    max_load_attempts = 1000\n\n    def __init__(self, engine_factory: Callable[[], Engine]) -> None:\n        self._engine_factory = engine_factory\n        self._engine: Engine | None = None\n        self._ready = threading.Event()\n        self._loading = False\n        self._state_lock = threading.Lock()\n        self._generate_lock = threading.Lock()\n        self.device = "cpu"\n\n    # -- loading ---------------------------------------------------------------\n\n    @property\n    def loaded(self) -> bool:\n        return self._ready.is_set()\n\n    @property\n    def sample_rate(self) -> int:\n        return self._engine.sample_rate if self._engine else 24_000\n\n    def start_loading(self) -> None:\n        with self._state_lock:\n            if self._ready.is_set() or self._loading:\n                return\n            self._loading = True\n        threading.Thread(target=self._load, name="model-loader", daemon=True).start()\n\n    def _load(self) -> None:\n        # Retries with backoff: on a slow or flaky network the first-run weight\n        # download (~3.2 GB) can fail part-way; huggingface_hub resumes it.\n        delay = self.retry_initial_seconds\n        attempt = 0\n        try:\n            while True:\n                attempt += 1\n                started = time.monotonic()\n                logger.info("model loading started (attempt %d)", attempt)\n                try:\n                    engine = self._engine_factory()\n                    engine.load()\n                except Exception:\n                    logger.exception(\n                        "model failed to load after %.1fs; retrying in %.0fs",\n                        time.monotonic() - started, delay,\n                    )\n                    if attempt >= self.max_load_attempts:\n                        return\n                    time.sleep(delay)\n                    delay = min(delay * 2, 300.0)\n                    continue\n                self._engine = engine\n                self.device = getattr(engine, "device", "cpu")\n                self._ready.set()\n                logger.info("model loaded in %.1fs", time.monotonic() - started)\n                return\n        finally:\n            with self._state_lock:\n                self._loading = False\n\n    def wait_ready(self, timeout: float) -> bool:\n        if self._ready.is_set():\n            return True\n        # A failed load is retried on demand.\n        self.start_loading()\n        return self._ready.wait(timeout)\n\n    # -- generation --------------------------------------------------------------\n\n    def forget_voice(self, voice_id: str) -> None:\n        if self._engine is not None:\n            with self._generate_lock:\n                self._engine.forget_voice(voice_id)\n\n    def synthesize(\n        self,\n        text: str,\n        language_id: str,\n        voice_id: str,\n        voice_path: Path,\n        cancel: threading.Event,\n    ) -> np.ndarray:\n        """Generate speech sentence by sentence; blocks, so call it from a worker thread."""\n        if not self._ready.is_set() or self._engine is None:\n            raise ModelNotReady()\n        chunks = split_sentences(text)\n        sr = self._engine.sample_rate\n        gap = np.zeros(int(SENTENCE_GAP_SECONDS * sr), dtype=np.float32)\n        parts: list[np.ndarray] = []\n        started = time.monotonic()\n        with self._generate_lock:\n            for index, chunk in enumerate(chunks):\n                if cancel.is_set():\n                    raise SynthesisCancelled()\n                t0 = time.monotonic()\n                wav = self._engine.generate(chunk, language_id, voice_id, voice_path)\n                logger.info(\n                    "chunk %d/%d: %d chars -> %.2fs audio in %.2fs",\n                    index + 1, len(chunks), len(chunk), wav.size / sr, time.monotonic() - t0,\n                )\n                if parts:\n                    parts.append(gap)\n                parts.append(np.asarray(wav, dtype=np.float32).reshape(-1))\n        audio = np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)\n        logger.info(\n            "synthesized %d chars (%d chunks, lang=%s) -> %.2fs audio in %.2fs",\n            len(text), len(chunks), language_id, audio.size / sr, time.monotonic() - started,\n        )\n        return audio\n',
    'audio.py': '"""Audio helpers: decoding uploads, building the reference clip, WAV encoding.\n\nOnly numpy + soundfile are required. ``soxr`` (installed in the Docker image\nas a librosa dependency) is used for high-quality resampling when available;\notherwise a linear-interpolation fallback keeps the tests dependency-free.\n"""\n\nfrom __future__ import annotations\n\nimport io\nimport re\n\nimport numpy as np\nimport soundfile as sf\n\n# Chatterbox\'s S3Gen decoder works at 24 kHz and resamples the reference to it.\nREFERENCE_SAMPLE_RATE = 24_000\nMIN_REFERENCE_SECONDS = 5.0\nMAX_REFERENCE_SECONDS = 20.0\n# When there is more voiced audio than MAX_REFERENCE_SECONDS, keep the most\n# voiced window of this length.\nTARGET_REFERENCE_SECONDS = 15.0\n\n_FRAME_SECONDS = 0.02\n_SILENCE_DB_BELOW_PEAK = 40.0\n_ABS_SILENCE_RMS = 1e-4\n_MAX_INNER_GAP_SECONDS = 0.3\n_FILE_JOIN_GAP_SECONDS = 0.15\n\n\nclass AudioError(ValueError):\n    """Raised for audio the service cannot use (maps to HTTP 400)."""\n\n\ndef decode_audio(data: bytes) -> tuple[np.ndarray, int]:\n    """Decode WAV (or FLAC/OGG/MP3 when libsndfile supports it) to mono float32."""\n    if not data:\n        raise AudioError("empty audio file")\n    try:\n        samples, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)\n    except Exception as exc:  # soundfile raises several error types\n        raise AudioError("could not decode audio (send WAV)") from exc\n    if samples.size == 0 or sr <= 0:\n        raise AudioError("empty audio file")\n    mono = samples.mean(axis=1).astype(np.float32)\n    if not np.all(np.isfinite(mono)):\n        mono = np.nan_to_num(mono, nan=0.0, posinf=0.0, neginf=0.0)\n    return mono, int(sr)\n\n\ndef resample(samples: np.ndarray, orig_sr: int, target_sr: int) -> np.ndarray:\n    if orig_sr == target_sr or samples.size == 0:\n        return samples.astype(np.float32, copy=False)\n    try:\n        import soxr  # type: ignore\n\n        return soxr.resample(samples, orig_sr, target_sr, quality="HQ").astype(np.float32)\n    except ImportError:\n        n_out = int(round(samples.size * target_sr / orig_sr))\n        x_old = np.arange(samples.size, dtype=np.float64) / orig_sr\n        x_new = np.arange(n_out, dtype=np.float64) / target_sr\n        return np.interp(x_new, x_old, samples).astype(np.float32)\n\n\ndef _frame_rms(samples: np.ndarray, frame: int) -> np.ndarray:\n    n_frames = samples.size // frame\n    if n_frames == 0:\n        return np.zeros(0, dtype=np.float32)\n    frames = samples[: n_frames * frame].reshape(n_frames, frame)\n    return np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1)).astype(np.float32)\n\n\ndef _voiced_mask(rms: np.ndarray) -> np.ndarray:\n    if rms.size == 0:\n        return np.zeros(0, dtype=bool)\n    peak = float(rms.max())\n    threshold = max(peak * 10 ** (-_SILENCE_DB_BELOW_PEAK / 20), _ABS_SILENCE_RMS)\n    return rms > threshold\n\n\ndef _compact_silence(samples: np.ndarray, sr: int) -> np.ndarray:\n    """Trim leading/trailing silence and shorten long internal pauses."""\n    frame = max(1, int(sr * _FRAME_SECONDS))\n    rms = _frame_rms(samples, frame)\n    voiced = _voiced_mask(rms)\n    if not voiced.any():\n        return np.zeros(0, dtype=np.float32)\n    max_gap = max(1, int(_MAX_INNER_GAP_SECONDS / _FRAME_SECONDS))\n    first, last = int(np.argmax(voiced)), int(len(voiced) - np.argmax(voiced[::-1]) - 1)\n    keep: list[np.ndarray] = []\n    gap = 0\n    for i in range(first, last + 1):\n        if voiced[i]:\n            gap = 0\n        else:\n            gap += 1\n            if gap > max_gap:\n                continue\n        keep.append(samples[i * frame : (i + 1) * frame])\n    return np.concatenate(keep).astype(np.float32)\n\n\ndef _best_window(samples: np.ndarray, sr: int, seconds: float) -> np.ndarray:\n    """Return the window of `seconds` with the most voiced energy."""\n    win = int(seconds * sr)\n    if samples.size <= win:\n        return samples\n    frame = max(1, int(sr * _FRAME_SECONDS))\n    rms = _frame_rms(samples, frame)\n    energy = np.where(_voiced_mask(rms), rms, 0.0).astype(np.float64)\n    win_frames = max(1, win // frame)\n    if energy.size <= win_frames:\n        return samples[:win]\n    csum = np.concatenate([[0.0], np.cumsum(energy)])\n    scores = csum[win_frames:] - csum[:-win_frames]\n    start = int(np.argmax(scores)) * frame\n    return samples[start : start + win]\n\n\ndef build_reference(clips: list[tuple[np.ndarray, int]]) -> np.ndarray:\n    """Combine uploaded clips into one clean reference at REFERENCE_SAMPLE_RATE."""\n    sr = REFERENCE_SAMPLE_RATE\n    gap = np.zeros(int(_FILE_JOIN_GAP_SECONDS * sr), dtype=np.float32)\n    parts: list[np.ndarray] = []\n    for samples, clip_sr in clips:\n        compact = _compact_silence(resample(samples, clip_sr, sr), sr)\n        if compact.size:\n            if parts:\n                parts.append(gap)\n            parts.append(compact)\n    combined = np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)\n    if combined.size < MIN_REFERENCE_SECONDS * sr:\n        raise AudioError(\n            f"not enough speech: need at least {MIN_REFERENCE_SECONDS:.0f} seconds of non-silent audio"\n        )\n    if combined.size > MAX_REFERENCE_SECONDS * sr:\n        combined = _best_window(combined, sr, TARGET_REFERENCE_SECONDS)\n    peak = float(np.max(np.abs(combined)))\n    if peak > 0:\n        combined = combined * (0.89 / peak)  # about -1 dBFS\n    return combined.astype(np.float32)\n\n\ndef encode_wav(samples: np.ndarray, sr: int) -> bytes:\n    buf = io.BytesIO()\n    sf.write(buf, np.clip(samples, -1.0, 1.0), sr, format="WAV", subtype="PCM_16")\n    return buf.getvalue()\n\n\n# --- text --------------------------------------------------------------------\n\n_SENTENCE_END = re.compile(r"(?<=[.!?।॥。！？؟])\\s+|(?<=[。！？])|\\n+")\n_MAX_CHUNK_CHARS = 250\n\n\ndef split_sentences(text: str, max_chars: int = _MAX_CHUNK_CHARS) -> list[str]:\n    """Split text into sentence-sized chunks of at most ``max_chars`` characters.\n\n    Short sentences are merged so that the model is not called for every\n    fragment; overly long sentences are split at commas, then at spaces.\n    """\n    pieces = [p.strip() for p in _SENTENCE_END.split(text) if p and p.strip()]\n    sentences: list[str] = []\n    for piece in pieces:\n        if len(piece) <= max_chars:\n            sentences.append(piece)\n            continue\n        current = ""\n        for part in re.split(r"(?<=[,，、;:])\\s*|\\s+", piece):\n            if not part:\n                continue\n            while len(part) > max_chars:  # no break point at all\n                if current:\n                    sentences.append(current)\n                    current = ""\n                sentences.append(part[:max_chars])\n                part = part[max_chars:]\n            candidate = f"{current} {part}".strip() if current else part\n            if len(candidate) > max_chars:\n                sentences.append(current)\n                current = part\n            else:\n                current = candidate\n        if current:\n            sentences.append(current)\n\n    chunks: list[str] = []\n    for sentence in sentences:\n        if chunks and len(chunks[-1]) + 1 + len(sentence) <= max_chars // 2:\n            chunks[-1] = f"{chunks[-1]} {sentence}"\n        else:\n            chunks.append(sentence)\n    return chunks\n',
}
for name, text in FILES.items():
    with open(f"/content/vc/{name}", "w", encoding="utf-8") as fh:
        fh.write(text)
print("Code likh diya:", ", ".join(FILES))

In [ ]:
# 4) Service chalana + public link (token zaroori: iske baghair koi bhi link use kar sakta hai)
# Khud-mukhtar: Colab ne beech mein session restart kiya ho tab bhi chalta hai.
import getpass, json, os, re, subprocess, time, urllib.request

ENGINE = globals().get("ENGINE", "omnivoice")
VOICES = "/content/drive/MyDrive/ai-clone-voices"
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(VOICES, exist_ok=True)
if not os.path.isfile("/content/vc/app.py"):
    raise SystemExit("Code nahi mila: pehle cell 3 chalayen (ya Runtime > Run all).")

token = getpass.getpass("Token paste karein (Ctrl+V, phir Enter): ").strip()
if len(token) < 16:
    raise SystemExit("Token khaali ya ghalat hai. Laptop par scripts\\voice-gpu.ps1 dobara chalayen, wo token copy karti hai.")

env = dict(os.environ, VOICE_CLONE_SERVICE_TOKEN=token, VOICE_DATA_DIR=VOICES, VOICE_CLONE_DEVICE="cuda",
           VOICE_CLONE_ENGINE=ENGINE, OMNIVOICE_STEPS="32",
           SYNTH_TIMEOUT_SECONDS="240", MODEL_WAIT_SECONDS="120", HF_HUB_DISABLE_TELEMETRY="1")
for name in ("server", "tunnel"):
    if name in globals():
        globals()[name].terminate()  # "Run all" again: stop the old ones first
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8100", "--workers", "1", "--no-access-log"],
    cwd="/content/vc", env=env, stdout=open("/content/vc/server.log", "w"), stderr=subprocess.STDOUT)

print("Model load ho raha hai (pehli baar ~3 GB download: 2-5 minute)...", flush=True)
for _ in range(180):
    time.sleep(10)
    if server.poll() is not None:
        print(open("/content/vc/server.log").read()[-4000:])
        raise SystemExit("Service band ho gayi (upar log dekhein).")
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:8100/health", timeout=5))
    except Exception:
        continue
    if health["model_loaded"]:
        print("Model tayyar:", health)
        break
else:
    raise SystemExit("30 minute mein model load nahi hua.")

if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8100"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    time.sleep(2)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found.group(0)
        break
if not url:
    print(open("/content/tunnel.log").read()[-3000:])
    raise SystemExit("Public link nahi bana (upar log dekhein). Run all dobara karein.")
print("\n" + "=" * 70)
print("YE LINK COPY KAR KE LAPTOP WALI SCRIPT MEIN PASTE KAREIN:\n")
print("   ", url)
print("=" * 70)

In [ ]:
# 5) Chalta rehne dein: har minute halat dikhata hai (Colab isse jaga rehta hai). Rokna ho to Stop dabayein.
import time
while True:
    time.sleep(60)
    alive = server.poll() is None and tunnel.poll() is None
    lines = [l for l in open("/content/vc/server.log").read().splitlines() if "synthesized" in l or "Error" in l]
    print(time.strftime("%H:%M"), "chal raha hai" if alive else "BAND HO GAYA: Run all dobara karein", "|", (lines[-1][-110:] if lines else ""))
    if not alive:
        break